<a target="_blank" href="https://colab.research.google.com/github/AshishKumar4/dew/blob/main/tutorials/04-samplers-and-schedules.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Samplers and schedules

[Notebook 02](02-train-a-diffusion-model.ipynb) trained a flowers DiT and sampled from it with one solver at one step count. This notebook loads that checkpoint and compares every sampler Dew ships: DDPM, DDIM, Euler, Euler ancestral, Heun, RK4 and MultiStepDPM, each at 10, 20 and 50 steps, from the same starting noise, with the wall time each one takes. It also shows what `get_diffusion_preset` pairs up, because the sampler only walks a trajectory the training schedule defined.

Run notebook 02 first: this one restores `./checkpoints/flowers-dit` from its working directory.

**Expected time**: about 5 minutes on any GPU or TPU, about 15 on a laptop CPU. It samples 84 grids of 16 images and nothing else.

In [ ]:
# On Colab: install dew and the JAX build for the runtime. Locally this cell is a no-op.
try:
    import google.colab  # noqa: F401
    import subprocess, sys
    try:
        import jax
        tpu = any("tpu" in str(d).lower() for d in jax.devices())
    except Exception:
        tpu = False
    extra = "jax[tpu]" if tpu else "jax[cuda12]"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "dew-ml[tfds] @ git+https://github.com/AshishKumar4/dew", extra, "matplotlib"])
except ImportError:
    pass

In [ ]:
# The whole run in one place.
CHECKPOINT = "checkpoints/flowers-dit"  # written by notebook 02
IMAGE_SIZE = 64
BATCH_SIZE = 64
MODEL = dict(patch_size=4, emb_features=384, num_layers=8, num_heads=6)
PROMPTS = None            # notebook 02's model is unconditional
SAMPLES = 16              # images per grid
STEP_COUNTS = (10, 20, 50)
SEED = 0

In [ ]:
import jax

print(jax.devices())
print(jax.default_backend())

## Rebuild the model and restore the checkpoint

A checkpoint holds parameters, not code. The model is rebuilt from the same config dictionary notebook 02 used, the preset is the same EDM pair, and `load_from_checkpoint` brings the weights back. What comes back is a `RestoredState`, which carries the saved arrays under the same attribute names a live train state uses, so `restored.params` and `restored.ema_params` are both there; the EMA weights are the ones to sample from.

In [ ]:
import jax.numpy as jnp
from dew.registry import apply_precision_policy, build_model
from dew.diffusion.transforms import get_diffusion_preset
from dew.inputs import DiffusionInputConfig
from dew.sampling.loading import load_from_checkpoint

train_schedule, sample_schedule, transform = get_diffusion_preset("edm")

model = build_model("simple_dit", apply_precision_policy(
    "simple_dit", MODEL, dtype="bfloat16", attention_impl="auto"))
inputs = DiffusionInputConfig(
    sample_data_key="image", sample_data_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), conditions=[])

restored = load_from_checkpoint(CHECKPOINT, step="best")
params = restored.ema_params
print(f"restored the EMA weights ({len(jax.tree_util.tree_leaves(params))} arrays) "
      f"from step {int(restored.step)}")

## The same starting noise for everyone

Every sampler integrates the same trajectory, so the only fair comparison fixes everything but the solver. The initial sample is drawn once, with the variance the schedule's top noise level implies, and handed to every run as `priors`. Ancestral samplers still inject their own noise along the way, which is part of what is being compared.

In [ ]:
alpha, sigma = sample_schedule.get_rates(sample_schedule.max_timesteps, shape=())
variance = jnp.sqrt(alpha ** 2 + sigma ** 2)
noise = jax.random.normal(jax.random.PRNGKey(SEED), (SAMPLES, IMAGE_SIZE, IMAGE_SIZE, 3))
priors = noise * variance
print(f"starting from noise scaled by sigma={float(sigma):.1f}")

## The solvers

- **DDPM** samples the reverse Markov chain the training schedule defines. It is the only sampler here whose recipe is fixed by the paper, and it needs small steps: at 10 or 20 it has not converged and shows streaks of unremoved noise.
- **DDIM** is DDPM's deterministic shortcut: jump to any next noise level, keep the direction the model implies. It gives the cheap baseline the others are judged against, and at 10 steps it is usually the best of the first-order solvers.
- **Euler** integrates the probability-flow ODE with one evaluation per step. On the Karras schedule it lands very close to DDIM, which is expected: DDIM is exactly Euler on that ODE.
- **Euler ancestral** solves the reverse SDE: after every deterministic step it injects a calibrated fraction of fresh noise. It is the sampler notebook 02 used; the stochasticity cleans up what a coarse ODE step leaves behind, at the price of a different sample every run.
- **Heun** is Euler with a correction: evaluate at the end of the step, average the two slopes. Two model evaluations per step buy roughly second-order accuracy, so 20 Heun steps usually beat 40 Euler ones, at the same number of evaluations as 40.
- **RK4** is the classical fourth-order Runge-Kutta: four evaluations per step for a step size one order better still. It is the accuracy ceiling here, and the most expensive per step.
- **MultiStepDPM** reuses predictions from earlier steps to build a second and third order correction without extra evaluations. After the first step or two it is nearly free accuracy, which is why production samplers look like it.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from dew.sampling import (DDPMSampler, DDIMSampler, EulerSampler, EulerAncestralSampler,
                          HeunSampler, RK4Sampler, MultiStepDPM)

SAMPLERS = {
    "DDPM": DDPMSampler,
    "DDIM": DDIMSampler,
    "Euler": EulerSampler,
    "Euler-A": EulerAncestralSampler,
    "Heun": HeunSampler,
    "RK4": RK4Sampler,
    "MultiStepDPM": MultiStepDPM,
}

def save_grid(rows, labels, path):
    '''rows: list of [n, H, W, 3] float arrays in [-1, 1]; one grid row per sampler.'''
    frames = [np.clip((np.asarray(r) + 1) * 127.5, 0, 255).astype(np.uint8) for r in rows]
    import os
    os.makedirs(os.path.dirname(path), exist_ok=True)
    grid = np.concatenate([np.concatenate(f, axis=1) for f in frames], axis=0)
    Image.fromarray(grid).save(path)
    plt.figure(figsize=(frames[0].shape[0] * 1.0, len(frames) * 1.0))
    plt.imshow(grid)
    plt.title(" | ".join(labels))
    plt.axis("off")
    plt.show()
    return path

timings = {}
for n_steps in STEP_COUNTS:
    rows, labels = [], []
    for name, cls in SAMPLERS.items():
        sampler = cls(model, sample_schedule, transform, inputs)
        start = time.perf_counter()
        images = sampler.generate_samples(params=params, num_samples=SAMPLES,
                                          resolution=IMAGE_SIZE, diffusion_steps=n_steps,
                                          priors=priors)
        timings[(name, n_steps)] = time.perf_counter() - start
        rows.append(images)
        labels.append(name)
    save_grid(rows, labels, f"samples/04-{n_steps}-steps.png")
    print(f"{n_steps} steps done")

In [ ]:
def fmt(seconds):
    return f"{seconds:.2f}s"

print(f"{'sampler':<13}" + "".join(f"{n:>9}" for n in STEP_COUNTS))
for name in SAMPLERS:
    row = "".join(f"{fmt(timings[(name, n)]):>9}" for n in STEP_COUNTS)
    print(f"{name:<13}{row}")
evals = {"DDPM": 1, "DDIM": 1, "Euler": 1, "Euler-A": 1, "Heun": 2, "RK4": 4, "MultiStepDPM": 1}
print()
print(f"{'evals/step':<13}" + "".join(f"{evals[name]:>9}" for name in SAMPLERS))

## What the grids say

The pattern to look for, at each step count: the first-order solvers (DDPM, DDIM, Euler, Euler-A) sharpen together as steps grow, with DDPM lagging at 10 because its variance schedule wants small steps. Heun and RK4 are already clean at 20. MultiStepDPM tracks them at nearly the first-order wall time, which is the whole point of reusing history. Euler and DDIM land within noise of each other at every count, the algebraic equivalence showing through.

The wall-time column mostly measures evaluations per step: RK4 costs about four Euler runs, Heun two, MultiStepDPM about one.

## The presets behind the pairings

A training schedule and a sampling schedule are not independent choices. `get_diffusion_preset` returns the matched pair with the prediction transform that binds them: EDM trains on a log-normal sigma distribution and samples on the Karras spacing; the cosine preset trains and samples a discrete variance-preserving schedule with v-prediction; the flow preset is rectified flow on both sides. Sampling a model with a different preset than it trained under silently produces the wrong trajectory, which is why there is one function that returns both.

In [ ]:
for name in ("edm", "karras", "cosine", "flow"):
    train, sample, prediction = get_diffusion_preset(name)
    print(f"{name:<8} train={type(train).__name__:<24} sample={type(sample).__name__:<24} "
          f"transform={type(prediction).__name__}")

## Where to go next

With a trained model, a checkpoint and a sense of which solver fits your budget, [notebook 03](03-text-to-image-with-guidance.ipynb) adds text conditioning and classifier-free guidance on top of the same machinery.